In [17]:
# chatbot/financial_intent.py
"""
Financial Intent Layer - Lớp chuyển dữ liệu thô thành insight có ngữ nghĩa
Mục tiêu: Tạo ra ngôn ngữ tự nhiên, hành động được, mang tính khuyên bảo cho user.
"""

from typing import Dict, Any, List, Optional

class FinancialIntentLayer:
    def __init__(self, analysis_data: Dict[str, Any]):
        self.data = analysis_data or {}
        self.categories: Dict = self.data.get('categories', {})
        self.summary: Dict = self.data.get('summary', {})

    # ====================== CONFIG ======================
    THRESHOLDS = {
        'food_stable_pct': 20,
        'food_stable_days': 20,
        'shopping_impulse_pct': 20,
        'shopping_impulse_max_days': 8,
        'ent_high_pct': 10,
        'ent_high_days': 15,
        'transport_high_pct': 15,
        'transport_high_days': 20,
        'saving_critical': 5,
        'saving_low': 15,
        'saving_good': 30,
        'volatility_high': 0.30,
        'volatility_stable': 0.15,
        'top_cat_alert': 25,
        'daily_high': 500_000,
        'daily_very_low': 100_000,
    }

    # Helper chung
    def _find_category(self, keywords: List[str]) -> Optional[Dict]:
        """Tìm category theo danh sách keywords"""
        for cat_name, info in self.categories.items():
            if any(kw.lower() in cat_name.lower() for kw in keywords):
                return info
        return None

    def detect_insights(self) -> List[str]:
        """Tự động chạy tất cả rules và trả về list insight"""
        insights = []
        for attr in dir(self):
            if attr.startswith('_rule_'):
                rule_func = getattr(self, attr)
                try:
                    result = rule_func()
                    if result:
                        insights.append(result)
                except Exception as e:
                    # TODO: logging
                    continue

        # Sắp xếp ưu tiên: Cảnh báo nguy hiểm lên đầu
        insights.sort(key=lambda x: (
            0 if any(w in x for w in ["CẢNH BÁO", "NGUY", "CHỈ SỐ NGUY"]) else
            1 if "⚠️" in x else 2
        ))
        return insights

    # ==================== RULES ====================

    def _rule_stable_food_spending(self):
        cat = self._find_category(['food', 'thực phẩm', 'ăn', 'restaurant', 'quán ăn', 'cafe', 'cà phê'])
        if not cat:
            return None
        t = self.THRESHOLDS
        if cat['pct_of_total'] > t['food_stable_pct'] and cat['active_days'] > t['food_stable_days']:
            return (f"✓ Chi tiêu ăn uống ổn định: chiếm {cat['pct_of_total']}% tổng chi, "
                    f"xuất hiện {cat['active_days']}/30 ngày. Đây là dấu hiệu thói quen lành mạnh.")
        return None

    def _rule_impulsive_shopping(self):
        cat = self._find_category(['shopping', 'mua sắm', 'fashion', 'thời trang', 'quần áo', 'shoes'])
        if not cat:
            return None
        t = self.THRESHOLDS
        if cat['pct_of_total'] > t['shopping_impulse_pct'] and cat['active_days'] < t['shopping_impulse_max_days']:
            return (f"⚠️ Mua sắm impulsively: {cat['pct_of_total']}% tổng chi nhưng chỉ trong {cat['active_days']} ngày. "
                    f"Đây là dấu hiệu chi tiêu bốc đồng. Nên lập danh sách mua sắm trước.")
        return None

    def _rule_entertainment_frequency(self):
        cat = self._find_category(['entertainment', 'giải trí', 'movie', 'phim', 'game', 'bar', 'pub', 'concert'])
        if not cat:
            return None
        t = self.THRESHOLDS
        if cat['pct_of_total'] > t['ent_high_pct'] and cat['active_days'] > t['ent_high_days']:
            return (f"ℹ️ Giải trí chiếm {cat['pct_of_total']}% tổng chi và xuất hiện {cat['active_days']} ngày/tháng. "
                    f"Bạn đang có thói quen giải trí thường xuyên. Hãy kiểm tra xem có đang dùng để giảm stress không.")
        return None

    def _rule_transport_efficiency(self):
        cat = self._find_category(['transport', 'taxi', 'grab', 'xăng', 'petrol', 'xe'])
        if not cat:
            return None
        t = self.THRESHOLDS
        if cat['pct_of_total'] > t['transport_high_pct'] and cat['active_days'] > t['transport_high_days']:
            return (f"💡 Chi phí vận chuyển cao ({cat['pct_of_total']}%). "
                    f"Xem xét tối ưu bằng phương tiện công cộng, đi bộ hoặc công ty hỗ trợ xe.")
        return None

    # ==================== SAVING RATE ====================
    def _rule_critical_saving_rate(self):
        saving = self.summary.get('saving_rate', 0)
        if saving < self.THRESHOLDS['saving_critical']:
            return (f"🚨 CẢNH BÁO NGHIÊM TRỌNG: Tỷ lệ tiết kiệm chỉ {saving}%. "
                    f"Bạn gần như không tiết kiệm được. Cần giảm mạnh chi tiêu ngay để tránh rủi ro tài chính.")
        return None

    def _rule_low_saving_rate(self):
        saving = self.summary.get('saving_rate', 0)
        if self.THRESHOLDS['saving_critical'] <= saving <= self.THRESHOLDS['saving_low']:
            return (f"🟡 Tỷ lệ tiết kiệm {saving}% vẫn còn thấp. "
                    f"Khuyến nghị nên hướng tới mức 20-30% để có nền tảng tài chính vững chắc.")
        return None

    def _rule_good_saving_rate(self):
        saving = self.summary.get('saving_rate', 0)
        if self.THRESHOLDS['saving_low'] < saving <= self.THRESHOLDS['saving_good']:
            return f"✅ Tỷ lệ tiết kiệm {saving}% ở mức tốt. Bạn đang kiểm soát tài chính khá ổn định."
        return None

    def _rule_excellent_saving_rate(self):
        saving = self.summary.get('saving_rate', 0)
        if saving > self.THRESHOLDS['saving_good']:
            return (f"🌟 Xuất sắc! Tỷ lệ tiết kiệm {saving}%. "
                    f"Bạn thuộc nhóm kiểm soát tài chính rất tốt.")
        return None

    # ==================== VOLATILITY & BALANCE ====================
    def _rule_high_spending_volatility(self):
        avg = self.summary.get('avg_monthly_expense', 0)
        std = self.summary.get('std_monthly_expense', 0)
        if avg > 0 and (std / avg) > self.THRESHOLDS['volatility_high']:
            return (f"📊 Chi tiêu biến động mạnh: độ lệch chuẩn {std:,.0f}đ "
                    f"({(std/avg*100):.1f}%). Nên xây dựng ngân sách cố định hàng tháng.")
        return None

    def _rule_deficit_spending(self):
        balance = self.summary.get('balance', 0)
        income = self.summary.get('total_income', 1)
        if balance < 0:
            deficit_pct = abs(balance) / income * 100
            return (f"🔴 NGUY HIỂM: Bạn chi tiêu vượt thu nhập {abs(balance):,.0f}đ "
                    f"({deficit_pct:.1f}%). Đây là tình trạng sống vượt khả năng tài chính.")
        return None

    def _rule_top_spending_category(self):
        if not self.categories:
            return None
        top = max(self.categories.items(), key=lambda x: x[1]['pct_of_total'])
        name, info = top
        if info['pct_of_total'] > self.THRESHOLDS['top_cat_alert']:
            return (f"📌 {name} là khoản chi lớn nhất ({info['pct_of_total']}% tổng chi). "
                    f"Đây là điểm đáng chú ý để tối ưu.")
        return None

    def _rule_daily_spending_average(self):
        daily = self.summary.get('daily_avg_expense', 0)
        if daily > self.THRESHOLDS['daily_high']:
            return (f"📈 Bạn chi trung bình {daily:,.0f}đ/ngày. "
                    f"Mức này khá cao, nên xem lại các khoản chi nhỏ lẻ.")
        elif daily < self.THRESHOLDS['daily_very_low']:
            return f"👍 Chi tiêu hàng ngày rất tiết kiệm ({daily:,.0f}đ/ngày)."
        return None

In [27]:
import pandas as pd
from datetime import datetime
from typing import Dict, Any, Optional

def aggregate_to_financial_analysis(
    df_income: pd.DataFrame, 
    df_expenses: pd.DataFrame, 
    account_id: str,
    reference_date: str = "2026-04-30"  # Dùng để tính active_days trong tháng
) -> Dict[str, Any]:
    """
    Aggregate dữ liệu từ Kaggle dataset cho một account (user) cụ thể
    Trả về định dạng đúng để đưa vào FinancialIntentLayer
    """
    
    # Lọc theo account
    income = df_income[df_income['account'] == account_id].copy()
    expenses = df_expenses[df_expenses['account'] == account_id].copy()
    
    if expenses.empty and income.empty:
        return {"summary": {}, "categories": {}}
    
    # Chuyển date_time sang datetime
    for df in [income, expenses]:
        if not df.empty:
            df['date'] = pd.to_datetime(df['date_time'])
    
    # ====================== SUMMARY ======================
    total_income = income['amount'].sum() if not income.empty else 0.0
    total_expense = expenses['amount'].sum() if not expenses.empty else 0.0
    balance = total_income - total_expense
    
    saving_rate = round((balance / total_income * 100), 1) if total_income > 0 else 0.0
    
    # Tính daily average (giả sử 30 ngày)
    daily_avg_expense = round(total_expense / 30, 0) if total_expense > 0 else 0
    
    # Tính std_monthly_expense (nếu có nhiều tháng)
    if not expenses.empty:
        expenses['month'] = expenses['date'].dt.to_period('M')
        monthly_exp = expenses.groupby('month')['amount'].sum()
        std_monthly = monthly_exp.std() if len(monthly_exp) > 1 else 0
    else:
        std_monthly = 0
    
    summary = {
        "total_income": round(total_income, 0),
        "total_expense": round(total_expense, 0),
        "balance": round(balance, 0),
        "saving_rate": saving_rate,
        "avg_monthly_expense": round(total_expense, 0),
        "std_monthly_expense": round(std_monthly, 0),
        "daily_avg_expense": daily_avg_expense
    }
    
    # ====================== CATEGORIES (chỉ tính chi tiêu) ======================
    categories = {}
    
    if not expenses.empty:
        cat_group = expenses.groupby('category').agg(
            amount=('amount', 'sum'),
            active_days=('date', 'nunique')
        ).reset_index()
        
        for _, row in cat_group.iterrows():
            cat_name = row['category']
            amount = row['amount']
            active_days = int(row['active_days'])
            
            pct_of_total = round((amount / total_expense * 100), 1) if total_expense > 0 else 0
            
            categories[cat_name] = {
                "pct_of_total": pct_of_total,
                "amount": round(amount, 0),
                "active_days": active_days
            }
    
    return {
        "summary": summary,
        "categories": categories
    }


# ====================== CÁCH SỬ DỤNG ======================

def get_all_users_analysis(df_income: pd.DataFrame, df_expenses: pd.DataFrame):
    """Tạo analysis cho tất cả accounts"""
    all_accounts = pd.concat([df_income['account'], df_expenses['account']]).unique()
    results = {}
    
    for acc in all_accounts:
        results[acc] = aggregate_to_financial_analysis(df_income, df_expenses, acc)
    
    return results

import numpy as np
from pprint import pformat

def clean_value(v):
    """Chuyển numpy types thành Python native và format số đẹp"""
    if isinstance(v, (np.integer, np.floating)):
        v = v.item()  # convert np.float64 / int64 thành Python float/int
    
    if isinstance(v, float):
        if v.is_integer():
            return int(v)
        else:
            return round(v, 1) if abs(v) < 1000 else round(v, 0)
    return v


def print_financial_analysis(data: dict):
    """In đẹp toàn bộ dữ liệu analysis"""
    for account, info in data.items():
        print("=" * 80)
        print(f"👤 ACCOUNT: {account}")
        print("=" * 80)
        
        summary = info.get('summary', {})
        categories = info.get('categories', {})
        
        print("📊 SUMMARY:")
        print("-" * 40)
        for key, value in summary.items():
            val = clean_value(value)
            if key == 'saving_rate':
                print(f"   {key:25}: {val:>8}%")
            elif 'amount' in key or 'expense' in key or 'balance' in key or 'income' in key:
                print(f"   {key:25}: {val:>12,} đ")
            else:
                print(f"   {key:25}: {val}")
        
        if categories:
            print("\n📂 CATEGORIES:")
            print("-" * 40)
            # Sắp xếp theo % giảm dần
            sorted_cats = sorted(
                categories.items(), 
                key=lambda x: clean_value(x[1]['pct_of_total']), 
                reverse=True
            )
            
            for cat_name, cat_info in sorted_cats:
                pct = clean_value(cat_info['pct_of_total'])
                amount = clean_value(cat_info['amount'])
                days = cat_info['active_days']
                print(f"   • {cat_name:25} | {amount:>10,} đ | {pct:5.1f}% | {days:3} ngày")
        else:
            print("\n📂 Không có chi tiêu nào.")
        
        print("\n")


In [ ]:
income_df = pd.read_csv('./data/Income_clean.csv')
expenses_df = pd.read_csv('./data/Expenses_clean.csv')

print("="*20 + " INCOME " + "="*20)
income_df.info()
print("\n\n" + "="*20 + " EXPENSES " + "="*20)
expenses_df.info()

In [28]:
analysis_data = get_all_users_analysis(income_df, expenses_df)
print_financial_analysis(analysis_data)

👤 ACCOUNT: acct_1
📊 SUMMARY:
----------------------------------------
   total_income             :       24,936 đ
   total_expense            :       13,064 đ
   balance                  :       11,872 đ
   saving_rate              :     47.6%
   avg_monthly_expense      :       13,064 đ
   std_monthly_expense      :          902 đ
   daily_avg_expense        :          435 đ

📂 CATEGORIES:
----------------------------------------
   • Loan given                |      2,809 đ |  21.5% |  10 ngày
   • Other                     |      2,465 đ |  18.9% |  11 ngày
   • Food                      |      1,322 đ |  10.1% |  72 ngày
   • Bought for myself         |      1,273 đ |   9.7% |   4 ngày
   • Gifts                     |      1,177 đ |   9.0% |  29 ngày
   • Cafe                      |      1,109 đ |   8.5% |  85 ngày
   • Health                    |        772 đ |   5.9% |  18 ngày
   • Leisure                   |        738 đ |   5.6% |  22 ngày
   • Taxi                      |    